In [16]:
import os
import pandas as pd
import numpy as np

# Setup relative paths
RAW_DATA_PATH = "../data/raw"
PROCESSED_DATA_PATH = "../data/processed"
os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

# Load the raw data downloaded from Step 1
wb_file = os.path.join(RAW_DATA_PATH, "singapore_worldbank_raw.csv")

if os.path.exists(wb_file):
    df = pd.read_csv(wb_file)
    print(f"[SUCCESS] Loaded raw data structure. Shape: {df.shape} (Rows, Columns)")
else:
    raise FileNotFoundError(f"Raw file not found at {wb_file}. Please run Notebook 01 first.")



[SUCCESS] Loaded raw data structure. Shape: (28, 10) (Rows, Columns)


In [17]:
# Order the historical timeline sequentially
df_sorted = df.sort_values("year").reset_index(drop=True)

# Execute time-series imputation (Forward fill then Backward fill to clear NaN gaps)
df_clean = df_sorted.ffill().bfill()

# Create a normalization helper function (scales variables between 0 and 1)
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min()) if (series.max() - series.min()) != 0 else 0

# --- UPDATED VALUE CONFIGURATIONS WITH DEBT METRICS ---
df_clean['scaled_growth'] = min_max_scale(df_clean['gdp_growth_annual_pct'])
df_clean['scaled_exports'] = min_max_scale(df_clean['exports_pct_gdp'])
df_clean['scaled_inflation'] = 1 - min_max_scale(df_clean['inflation_rate_pct']) # Lower inflation is better

# New Variable Scaling: High sovereign debt lowers the overall economic score
df_clean['scaled_debt'] = 1 - min_max_scale(df_clean['gov_debt_gdp_pct'])

# Recalculate your custom Economic Health Score with revised weights
df_clean['economic_health_score'] = (
    (df_clean['scaled_growth'] * 0.35) + 
    (df_clean['scaled_exports'] * 0.35) + 
    (df_clean['scaled_inflation'] * 0.15) +
    (df_clean['scaled_debt'] * 0.15)
) * 100

print("[SUCCESS] Score calculation updated to incorporate sovereign debt vulnerabilities.")


[SUCCESS] Score calculation updated to incorporate sovereign debt vulnerabilities.


In [18]:
# Explicitly define target file path
processed_file = os.path.join(PROCESSED_DATA_PATH, "singapore_processed_matrix.csv")

# Save cleaned dataframe
df_clean.to_csv(processed_file, index=False)

# Verification check to confirm it physically exists on your hard drive
if os.path.exists(processed_file):
    print(f"\n[SUCCESS] Matrix successfully written to disk!")
    print(f"File Path: {os.path.abspath(processed_file)}")
    print(f"Rows Saved: {len(df_clean)}")
else:
    print(f"\n[ERROR] File write failed. Check your workspace permissions.")



[SUCCESS] Matrix successfully written to disk!
File Path: D:\python\Python\data\processed\singapore_processed_matrix.csv
Rows Saved: 28
